# IRI v2 MPI Job

Runs a multi-node MPI program through the IRI Facility API **v2**. It is a normal compute job: the only difference is
`launcher: "mpiexec"` and more than one node. The flow follows [compute-jobs-v2.ipynb](compute-jobs-v2.ipynb):

1. **Build**: a normal single-node compute job writes a small MPI program (`mpi_hello.c`) into your job directory and
   compiles it with `mpicc`.
2. **Run**: an MPI job starts it on 2 nodes x 2 ranks. Every rank prints its host, the ranks do an `MPI_Allreduce`, and
   rank 0 measures latency and bandwidth to the last rank (on the other node).
3. **Output**: download the job's stdout with the filesystem API.

## How MPI jobs work at ESnet East

* Set `launcher` to `mpirun` or `mpiexec`. `processes_per_node` ranks run on each of `node_count` nodes;
  `process_count` (default `node_count * processes_per_node`) is the total.
* With `node_count` 1 the job runs `mpiexec -np <process_count>` in one container.
* With more nodes, all nodes start together in one availability zone and ranks are started **without ssh** (MPICH
  Hydra's manual launcher). The image must contain an **MPICH-family** MPI (MPICH, MVAPICH, or an MPICH-ABI build) and
  `bash`, and the program must be built against it. Open MPI and Intel MPI do not work across nodes here.
* Every rank's output arrives in node 0's stdout (`stdout_path`). Ranks talk TCP over the pod network (no EFA).
* At most 5 nodes per MPI job; `srun` is not available. The resource's `mpi_notes` attribute says the same.

This notebook uses the public `mpioperator/mpich-builder` (compile) and `mpioperator/mpich` (run) images, which carry
the same MPICH. Other facilities may start MPI differently; check their resource description first.

## Before you run
Use the [login-globus](login-globus.ipynb) or [login-esnet](login-esnet.ipynb) notebook to receive a token.


## 1) Imports & helper functions
Utilities used throughout the notebook.

In [ ]:
import os
import sys
import time
import uuid
import base64
import json
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
from dotenv import load_dotenv

load_dotenv()

def die(msg: str) -> None:
    print(f"\nERROR: {msg}")
    raise SystemExit(1)


def print_table(headers: List[str], rows: List[List[Any]]) -> None:
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    sep = "-+-".join("-" * w for w in widths)

    print(fmt.format(*headers))
    print(sep)
    for row in rows:
        print(fmt.format(*[str(c) for c in row]))
    print()


def cap_name(allcaps: Dict[str, dict], uri: str) -> str:
    c = allcaps.get(uri)
    return c.get("name") if c else uri


def human_size(n: Any) -> str:
    try:
        x = int(n)
    except Exception:
        return str(n)
    units = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    i = 0
    v = float(x)
    while v >= 1024.0 and i < len(units) - 1:
        v /= 1024.0
        i += 1
    if i == 0:
        return f"{int(v)} {units[i]}"
    return f"{v:.1f} {units[i]}"


def fmt_epoch(s: Any) -> str:
    """Convert epoch seconds string → UTC timestamp string."""
    try:
        e = int(str(s))
        return dt.datetime.fromtimestamp(e, tz=dt.UTC).strftime("%Y-%m-%d %H:%M:%SZ")
    except Exception:
        return str(s)

## 2) Configuration
Edit these variables (or use environment variables) to point at the right API and defaults.

In [ ]:
BASE_URL = "https://iri-dev.ppg.es.net/api/v2" # Esnet East
#BASE_URL = "https://esnet-west.sdn-sense.net/api/v2" # Esnet West
#BASE_URL = "https://api.iri.nersc.gov/api/v2" # NERSC
# ALCF and NRP serve IRI v1 only: use the v1 notebooks for them.

home_path = Path("~/.iri_token.json").expanduser()
TOKEN = os.environ.get("IRI_API_TOKEN")
if TOKEN:
    print(f"IMPORTANT: Token defined inside .env file (or as environment variable). WILL NOT LOAD TOKEN from {home_path}")
else:
    # Load token from home_path
    with open(home_path) as f:
        tokendict = json.load(f)
        TOKEN = tokendict.get("IRI_API_TOKEN")
# If the token is still not available. print warning!
if not TOKEN:
    print(f"WARNING! Was unable to receive token from env variable or {home_path}. Use login-globus or login-esnet notebook to receive a token.")
    print("WARNING! FURTHER COMMANDS WILL NOT SUCCEED.")

# Optional defaults for compute job. Unless set here (or in .env), the username comes
# from GET /account/whoami and the job directory from GET /storage/locations (both v2).
USERNAME = os.environ.get("IRI_USERNAME")
DEFAULT_JOB_DIR = os.environ.get("IRI_JOB_DIR")
DEFAULT_QUEUE = os.environ.get("IRI_QUEUE", "debug")
DEFAULT_ACCOUNT = os.environ.get("IRI_ACCOUNT", "interactive")

# v2: send an Idempotency-Key with the job submission, so a retried request cannot
# submit the job twice. Facilities without an idempotency store answer 400 naming the
# header (501 on older deployments); the job is not submitted, and the notebook then
# submits without the key.
USE_IDEMPOTENCY_KEY = os.environ.get("IRI_IDEMPOTENCY", "true").lower() == "true"

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}
POLL_INTERVAL = 5
# MPI jobs wait until every node has started, which can take a few minutes.
TIMEOUT = 900

## 3) Filesystem async-task helpers
Submit and poll filesystem tasks (`mkdir`, `ls`, `download`, etc.).

In v2 every filesystem operation is a `POST` with a JSON body; `fs_task` wraps submit + wait.

In [ ]:
def submit_fs_task(method: str, path: str, **kwargs) -> dict:
    url = f"{BASE_URL}{path}"
    print(f"Submitting {method} {url}")
    r = requests.request(method, url, headers=HEADERS, timeout=TIMEOUT, **kwargs)
    if not r.ok:
        die(f"{method} {url} failed: {r.status_code} {r.text}")

    data = r.json()
    if not data.get("task_id"):
        die(f"No task_id in response: {data}")
    if not data.get("task_uri"):
        die(f"No task_uri in response: {data}")
    return data


def wait_fs_task(taskin: dict) -> Any:
    deadline = time.time() + TIMEOUT
    while time.time() < deadline:
        r = requests.get(taskin["task_uri"], headers=HEADERS, timeout=TIMEOUT)
        if not r.ok:
            die(f"Task query failed: {r.status_code} {r.text}")

        t = r.json()
        status = t["status"]
        print(f"   FS Task {taskin['task_id']}: {status}")

        if status == "completed":
            return t.get("result")
        if status in ("failed", "canceled"):
            die(f"FS Task {taskin['task_id']} ended with status {status}: {t}")

        time.sleep(POLL_INTERVAL)

    die(f"FS Task {taskin['task_id']} timed out")


def fs_task(op: str, resource_id: str, **body) -> Any:
    """Run a v2 filesystem operation (POST /filesystem/<op>/<resource_id> with a JSON body) and wait for its result."""
    return wait_fs_task(submit_fs_task("POST", f"/filesystem/{op}/{resource_id}", json=body))

## 4) Discovery & resource selection
Lists projects/capabilities/resources and prompts you to pick **compute** and **filesystem** resources.

In [ ]:
def discover_resources_and_caps():
    print("\n" + "=" * 40)
    print("=== DISCOVERING RESOURCES AND CAPABILITIES ===")

    projects = requests.get(f"{BASE_URL}/account/projects", headers=HEADERS, timeout=TIMEOUT).json()
    project_rows = [[p.get("id"), p.get("name", ""), p.get("description", "")] for p in projects]
    print("\n=== PROJECTS ===")
    print_table(["Project ID", "Name", "Description"], project_rows)

    caps = requests.get(f"{BASE_URL}/account/capabilities", headers=HEADERS, timeout=TIMEOUT).json()
    cap_rows = [[c.get("self_uri"), c.get("name"), c.get("description", "")] for c in caps]
    cap_by_uri = {c["self_uri"]: c for c in caps}
    print("\n=== CAPABILITIES ===")
    print_table(["Capability URI", "Name", "Description"], cap_rows)

    alloc_rows = []
    project_caps = set()
    for pr in projects:
        allocs = requests.get(
            f"{BASE_URL}/account/projects/{pr['id']}/project_allocations",
            headers=HEADERS,
            timeout=TIMEOUT,
        ).json()
        for a in allocs:
            alloc_rows.append([pr["id"], a.get("id"), cap_name(cap_by_uri, a.get("capability_uri"))])
            if a.get("capability_uri") in cap_by_uri:
                project_caps.add(a["capability_uri"])

    print("\n=== PROJECT ALLOCATIONS ===")
    print_table(["Project ID", "Allocation ID", "Capability URI"], alloc_rows)

    if not project_caps:
        die("No allocations found for any project (need at least one capability).")

    resources = requests.get(
        f"{BASE_URL}/status/resources?offset=0&limit=100",
        headers=HEADERS,
        timeout=TIMEOUT,
    ).json()

    resource_rows = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        if any(cap in rcaps for cap in project_caps):
            cap_names = [cap_name(cap_by_uri, c) for c in rcaps]
            resource_rows.append(
                [
                    r.get("id"),
                    r.get("name", ""),
                    r.get("resource_type", ""),
                    r.get("description", ""),
                    ", ".join(cap_names),
                    r.get("current_status", ""),
                ]
            )

    print("\n=== RESOURCES (matching your project allocations) ===")
    print_table(
        ["Resource ID", "Name", "Type", "Description", "Capability URIs", "Current Status"],
        resource_rows,
    )

    return projects, cap_by_uri, resources, project_caps


def filter_resources_heuristic(resources, cap_by_uri, project_caps, kind: str):
    kind = kind.lower().strip()
    filtered = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        if not any(cap in rcaps for cap in project_caps):
            continue

        rtype = (r.get("resource_type") or "").lower()
        capnames = " ".join((cap_name(cap_by_uri, c) or "").lower() for c in rcaps)

        if kind == "compute":
            if "compute" in rtype or "job" in rtype or "slurm" in rtype or "compute" in capnames:
                filtered.append(r)
        elif kind == "filesystem":
            if (
                "storage" in rtype
                or "filesystem" in rtype
                or rtype == "fs"
                or "storage" in capnames
                or "filesystem" in capnames
            ):
                filtered.append(r)
        else:
            filtered.append(r)
    return filtered


# v2: resources advertise the operations they support as HAL _links (https://iri.science/rels/<rel>).
V2_RELS = {"compute": "iri:submit-job", "filesystem": "iri:list-directory"}


def v2_ops(resource: dict) -> str:
    """Which of the smoke test's operations a resource advertises in its v2 _links."""
    links = resource.get("_links") or {}
    return ", ".join(kind for kind, rel in V2_RELS.items() if rel in links)


def filter_resources(resources, cap_by_uri, project_caps, kind: str):
    """v2: prefer resources whose _links advertise the operation; fall back to the v1 name heuristics
    for facilities that do not publish them (e.g. NERSC compute resources have no iri:submit-job link)."""
    rel = V2_RELS.get(kind)
    by_link = [
        r for r in resources
        if rel in (r.get("_links") or {}) and any(c in (r.get("capability_uris") or []) for c in project_caps)
    ]
    return by_link or filter_resources_heuristic(resources, cap_by_uri, project_caps, kind)


def prompt_pick_resource(resources, cap_by_uri, title: str) -> str:
    rows = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        capnames = ", ".join(cap_name(cap_by_uri, c) for c in rcaps)
        rows.append([r.get("id"), r.get("name", ""), r.get("resource_type", ""), capnames, r.get("current_status", ""), v2_ops(r)])

    print("\n" + "=" * 40)
    print(f"=== {title} ===")
    print_table(["Resource ID", "Name", "Type", "Capabilities", "Status", "v2 ops"], rows)

    rid = input(f"Enter {title.lower()} resource ID:\n").strip()
    if not rid:
        die("No resource ID provided.")
    return rid

## 5) Compute job submission & status polling functions
Functions to submit the compute job and poll its status until completion.

`submit_compute_job` optionally sends an `Idempotency-Key` (v2). It returns the job id and the facility's
`Idempotency-Key-Reply` (`miss` = new submission, `hit` = replayed original response), or `None` when the key was not used.

In [ ]:
def submit_compute_job(compute_resource_id: str, payload: dict, idempotency_key: Optional[str] = None) -> tuple:
    url = f"{BASE_URL}/compute/job/{compute_resource_id}"
    print("\n" + "=" * 40)
    print(f"=== SUBMIT COMPUTE JOB: {url} ===")

    headers = {**HEADERS, "Content-Type": "application/json"}
    if idempotency_key:
        headers["Idempotency-Key"] = idempotency_key
        print(f"Idempotency-Key: {idempotency_key}")
    r = requests.post(url, headers=headers, json=payload, timeout=TIMEOUT)
    # 400 alone also covers unrelated request errors, so check that it names the header. Older deployments answer 501.
    key_unsupported = r.status_code == 501 or (r.status_code == 400 and "Idempotency-Key" in r.text)
    if key_unsupported and idempotency_key:
        # No idempotency store at this facility: the job was NOT submitted, so resending without the key is safe.
        print(f"Facility does not support Idempotency-Key ({r.status_code}); submitting without it.")
        idempotency_key = None
        r = requests.post(url, headers={**HEADERS, "Content-Type": "application/json"}, json=payload, timeout=TIMEOUT)
    if not r.ok:
        die(f"Compute submit failed: {r.status_code} {r.text}")

    data = r.json()
    job_id = data.get("id") or data.get("job_id")
    if not job_id:
        die(f"Compute submit response missing job id: {data}")

    reply = r.headers.get("Idempotency-Key-Reply") if idempotency_key else None
    print(f"Compute job submitted: job_id={job_id}" + (f" (Idempotency-Key-Reply: {reply})" if reply else ""))
    return job_id, reply


def poll_compute_status(compute_resource_id: str, job_id: str) -> dict:
    url = f"{BASE_URL}/compute/status/{compute_resource_id}/{job_id}"
    print("\n" + "=" * 40)
    print(f"=== POLLING COMPUTE STATUS: {url} ===")

    deadline = time.time() + TIMEOUT
    last = {}
    while time.time() < deadline:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if not r.ok:
            die(f"Compute status failed: {r.status_code} {r.text}")

        last = r.json()
        st = (last.get("status") or {})
        state = (st.get("state") or "").lower()
        msg = st.get("message", "")
        exit_code = st.get("exit_code", None)

        print(f"   job {job_id}: state={state} message={msg} exit_code={exit_code}")

        if state in {"completed", "failed", "canceled", "cancelled", "timeout", "timed_out"}:
            return last

        time.sleep(POLL_INTERVAL)

    die(f"Compute job {job_id} timed out while polling status.")

## 6) Log retrieval helpers
Pretty `ls` output + base64 decode for `download`.

In [ ]:
def render_ls_table(ls_result: Any, base_path: str) -> None:
    if not isinstance(ls_result, dict):
        print("LS result (unexpected type):", ls_result)
        return

    items = ls_result.get("output")
    if not isinstance(items, list):
        print("LS result (unexpected shape):", ls_result)
        return

    def key(it):
        t = (it.get("type") or "").lower()
        is_dir = 0 if t == "directory" else 1
        return (is_dir, (it.get("name") or "").lower())

    items_sorted = sorted(items, key=key)

    rows = []
    for it in items_sorted:
        rows.append(
            [
                it.get("type", ""),
                it.get("permissions", ""),
                it.get("user", ""),
                it.get("group", ""),
                human_size(it.get("size", "")),
                fmt_epoch(it.get("last_modified", "")),
                it.get("name", ""),
                it.get("link_target") or "",
            ]
        )

    print(f"\nListing: {base_path}")
    print_table(["Type", "Perm", "UID", "GID", "Size", "Last Modified (UTC)", "Name", "Link Target"], rows)


def extract_base64_payload(download_result: Any) -> Optional[str]:
    if not isinstance(download_result, dict):
        return None

    out = download_result.get("output")

    if isinstance(out, str):
        return out

    if isinstance(out, dict):
        for k in ("content_base64", "base64", "data_base64", "data", "content"):
            v = out.get(k)
            if isinstance(v, str) and v.strip():
                return v
    return None


def decode_base64_to_text(b64: str) -> str:
    try:
        raw = base64.b64decode(b64, validate=True)
    except Exception:
        raw = base64.b64decode(b64)

    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return raw.decode("utf-8", errors="replace")

## 7) Who am I (v2)
`GET /account/whoami` (new in v2) returns your username at the facility. It is used for the default job directory
below, instead of a hardcoded username. Set `IRI_USERNAME` to override it.

In [ ]:
r = requests.get(f"{BASE_URL}/account/whoami", headers=HEADERS, timeout=TIMEOUT)
if not r.ok:
    die(f"whoami failed: {r.status_code} {r.text}")
whoami = r.json()
print("Facility username (whoami):", whoami.get("username"))
if not USERNAME:
    USERNAME = whoami["username"]
print("USERNAME:", USERNAME)

## 8) Pick resources
Identify available resources and select one for the compute and filesystem


In [ ]:
projects, cap_by_uri, resources, project_caps = discover_resources_and_caps()

compute_candidates = filter_resources(resources, cap_by_uri, project_caps, "compute")
fs_candidates = filter_resources(resources, cap_by_uri, project_caps, "filesystem")

if not compute_candidates:
    print("\n(No compute candidates matched heuristics; showing all matching resources instead.)")
    compute_candidates = [r for r in resources if any(c in (r.get("capability_uris") or []) for c in project_caps)]

if not fs_candidates:
    print("\n(No filesystem candidates matched heuristics; showing all matching resources instead.)")
    fs_candidates = [r for r in resources if any(c in (r.get("capability_uris") or []) for c in project_caps)]

COMPUTE_RESOURCE_ID = prompt_pick_resource(compute_candidates, cap_by_uri, "COMPUTE RESOURCE")
FS_RESOURCE_ID = prompt_pick_resource(fs_candidates, cap_by_uri, "FILESYSTEM RESOURCE")

print("\nChosen compute resource:", COMPUTE_RESOURCE_ID)
print("Chosen filesystem resource:", FS_RESOURCE_ID)

## 9) Resolve the job directory (v2)
`GET /storage/locations/{resource_id}?intent=write` (new in v2) returns the directories you can write to on a resource
(home, project, scratch, ...), resolved for your user. The job directory is your writable home on the chosen filesystem
resource (or the compute resource). Set `IRI_JOB_DIR` to override it.

In [ ]:
def resolve_writable_dir(resource_ids: List[str]) -> Optional[str]:
    """Writable directory for this user from GET /storage/locations (v2): home first, else any writable location."""
    for rid in resource_ids:
        r = requests.get(f"{BASE_URL}/storage/locations/{rid}", headers=HEADERS, params={"intent": "write"}, timeout=TIMEOUT)
        if not r.ok:
            print(f"   storage/locations/{rid}: {r.status_code} {r.text[:200]}")
            continue
        locations = r.json()
        print(f"\n=== STORAGE LOCATIONS ({rid}) ===")
        print_table(
            ["Logical name", "Path", "Filesystem", "Performance", "Shared", "Read", "Write"],
            [
                [loc.get("logical_name"), loc.get("path"), loc.get("filesystem") or "", loc.get("performance_tier") or "",
                 loc.get("shared"), (loc.get("access") or {}).get("read"), (loc.get("access") or {}).get("write")]
                for loc in locations
            ],
        )
        writable = [loc for loc in locations if (loc.get("access") or {}).get("write")]
        for loc in sorted(writable, key=lambda loc: loc.get("logical_name") != "home"):
            return loc["path"]
    return None


if DEFAULT_JOB_DIR:
    print(f"Using IRI_JOB_DIR={DEFAULT_JOB_DIR}")
else:
    DEFAULT_JOB_DIR = resolve_writable_dir([FS_RESOURCE_ID, COMPUTE_RESOURCE_ID])
    if not DEFAULT_JOB_DIR:
        DEFAULT_JOB_DIR = f"/data/home/{USERNAME}"
        print(f"No writable location from storage/locations; falling back to {DEFAULT_JOB_DIR} (set IRI_JOB_DIR to override).")
print("Job directory:", DEFAULT_JOB_DIR)

## 10) Build the MPI program (a normal compute job)
One node, no launcher: write `mpi_hello.c` into `<job dir>/mpi-example/` and compile it with `mpicc`.


In [ ]:
import shlex

MPI_SOURCE = r"""
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char **argv) {
  int rank, size, len;
  char host[MPI_MAX_PROCESSOR_NAME];
  MPI_Init(&argc, &argv);
  MPI_Comm_rank(MPI_COMM_WORLD, &rank);
  MPI_Comm_size(MPI_COMM_WORLD, &size);
  MPI_Get_processor_name(host, &len);
  printf("rank %d of %d on %s\n", rank, size, host);
  fflush(stdout);

  long mine = rank + 1, sum = 0;
  MPI_Allreduce(&mine, &sum, 1, MPI_LONG, MPI_SUM, MPI_COMM_WORLD);
  if (rank == 0) printf("allreduce sum %ld (expected %ld)\n", sum, (long)size * (size + 1) / 2);

  /* ping-pong between rank 0 and the last rank, which sits on the other node */
  const int peer = size - 1, big = 8 << 20;
  char *buf = calloc(big, 1);
  MPI_Barrier(MPI_COMM_WORLD);
  if (peer > 0 && (rank == 0 || rank == peer)) {
    int other = rank == 0 ? peer : 0;
    for (int k = 0; k < 2; k++) {
      int bytes = k ? big : 8, reps = k ? 20 : 1000;
      double t = MPI_Wtime();
      for (int i = 0; i < reps; i++) {
        if (rank == 0) {
          MPI_Send(buf, bytes, MPI_CHAR, other, 0, MPI_COMM_WORLD);
          MPI_Recv(buf, bytes, MPI_CHAR, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        } else {
          MPI_Recv(buf, bytes, MPI_CHAR, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
          MPI_Send(buf, bytes, MPI_CHAR, other, 0, MPI_COMM_WORLD);
        }
      }
      t = MPI_Wtime() - t;
      if (rank == 0 && k) printf("bandwidth rank 0 <-> %d, 8 MiB messages: %.1f MB/s\n", peer, 2.0 * bytes * reps / t / 1e6);
      if (rank == 0 && !k) printf("latency rank 0 <-> %d, 8 B: %.1f us\n", peer, t / reps / 2 * 1e6);
    }
  }
  free(buf);
  MPI_Finalize();
  return 0;
}
"""

timestamp = dt.datetime.now(dt.UTC).strftime("%Y%m%d-%H%M%S")
mpi_dir = f"{DEFAULT_JOB_DIR.rstrip('/')}/mpi-example"
mpi_binary = f"{mpi_dir}/mpi_hello"

# Not an f-string: the C source has braces. The quoted heredoc keeps it verbatim.
build_script = (
    "set -euo pipefail\n"
    f"mkdir -p {shlex.quote(mpi_dir)}\n"
    f"cat > {shlex.quote(mpi_dir)}/mpi_hello.c <<'MPI_SOURCE_END'\n" + MPI_SOURCE + "MPI_SOURCE_END\n"
    f"mpicc -O2 -o {shlex.quote(mpi_binary)} {shlex.quote(mpi_dir)}/mpi_hello.c\n"
    f"ls -l {shlex.quote(mpi_dir)}\n"
)

build_payload = {
    "name": f"iri-v2-mpi-build-{timestamp}",
    "executable": "bash",
    "arguments": ["-lc", build_script],
    "directory": DEFAULT_JOB_DIR,
    "container": {"image": "mpioperator/mpich-builder:v0.8.2"},
    "stdout_path": f"{DEFAULT_JOB_DIR.rstrip('/')}/iri_v2_mpi_build_stdout_{timestamp}.log",
    "stderr_path": f"{DEFAULT_JOB_DIR.rstrip('/')}/iri_v2_mpi_build_stderr_{timestamp}.log",
    "resources": {"node_count": 1, "processes_per_node": 1, "cpu_cores_per_process": 1, "memory": 1073741824},
    "attributes": {"duration": 600, "queue_name": DEFAULT_QUEUE, "account": DEFAULT_ACCOUNT, "custom_attributes": {}},
}

build_job_id, _ = submit_compute_job(COMPUTE_RESOURCE_ID, build_payload, str(uuid.uuid4()) if USE_IDEMPOTENCY_KEY else None)
build_status = poll_compute_status(COMPUTE_RESOURCE_ID, build_job_id)
build_state = ((build_status.get("status") or {}).get("state") or "").lower()
if build_state != "completed":
    die(f"Build job did not complete (state={build_state}); see {build_payload['stderr_path']}")
print("Built:", mpi_binary)


## 11) Submit the MPI job
The same request as any compute job, plus `launcher: "mpiexec"` and `node_count` above 1: 2 nodes x 2 ranks = 4 ranks.
The image only has to provide MPICH's `mpiexec`; the facility starts the ranks on every node.


In [ ]:
mpi_stdout_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/iri_v2_mpi_stdout_{timestamp}.log"
mpi_stderr_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/iri_v2_mpi_stderr_{timestamp}.log"

mpi_payload = {
    "name": f"iri-v2-mpi-{timestamp}",
    "executable": mpi_binary,
    "arguments": [],
    "launcher": "mpiexec",
    "directory": mpi_dir,
    "container": {"image": "mpioperator/mpich:v0.8.2"},
    "stdout_path": mpi_stdout_path,
    "stderr_path": mpi_stderr_path,
    "resources": {
        "node_count": 2,
        "processes_per_node": 2,
        "process_count": 4,
        "cpu_cores_per_process": 1,
        "memory": 536870912,  # per process
    },
    "attributes": {"duration": 900, "queue_name": DEFAULT_QUEUE, "account": DEFAULT_ACCOUNT, "custom_attributes": {}},
}

IDEMPOTENCY_KEY = str(uuid.uuid4()) if USE_IDEMPOTENCY_KEY else None
job_id, IDEMPOTENCY_REPLY = submit_compute_job(COMPUTE_RESOURCE_ID, mpi_payload, IDEMPOTENCY_KEY)


## 12) Poll job status
`pending` covers waiting in the queue and waiting for every node to start.


In [ ]:
final_status = poll_compute_status(COMPUTE_RESOURCE_ID, job_id)

state = ((final_status.get("status") or {}).get("state") or "").lower()
exit_code = (final_status.get("status") or {}).get("exit_code", None)

print("\n" + "=" * 40)
print("=== MPI JOB FINISHED ===")
print(f"job_id   : {job_id}")
print(f"state    : {state}")
print(f"exit_code: {exit_code}")
print("=" * 40)


## 13) Download the MPI output
All ranks' output is in node 0's stdout. If the job failed, stderr says why (for example a non-MPICH image).


In [ ]:
for label, path in (("STDOUT", mpi_stdout_path), ("STDERR", mpi_stderr_path)):
    print("\n" + "=" * 40)
    print(f"=== FILESYSTEM: DOWNLOAD {label} ===")
    result = fs_task("download", FS_RESOURCE_ID, path=path)
    b64 = extract_base64_payload(result)
    print(decode_base64_to_text(b64) if b64 else f"(no content: {result})")

if state != "completed" or (exit_code not in (None, 0)):
    die(f"MPI job did not complete cleanly (state={state}, exit_code={exit_code}).")
